# <center> <font color="#0036a3">Maestría en Inteligencia Artificial Aplicada (MNA)</font>  — Avance 10</center>

## **<font color="#0036a3">Avance 10 — App web 3D interactiva: reconstrucción con/sin enhancement</font>**

### **<font color="#E0A800">Proyecto Integrador — TC5035.10 · Equipo 52</font>**

---

Genera **nubes de puntos 3D** (formato JSON) para las **7 escenas nuevas** del Avance9, con dos versiones cada una:
- **Baseline (none)**: profundidad predicha sin enhancement.
- **Enhanced (IAT)**: profundidad predicha sobre imagen realzada.

Cada nube incluye **posición XYZ** y **color RGB**, lista para renderizar en web con **Three.js**. La app web permitirá visualización lado-a-lado interactiva con lupa simultánea y rotaciones fijas.

In [ ]:
from pathlib import Path
import sys, subprocess

IN_COLAB = "google.colab" in sys.modules
if not IN_COLAB:
    try:
        import google.colab; IN_COLAB = True
    except ImportError:
        IN_COLAB = False

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
    BASE         = Path("/content/drive/MyDrive/proyecto_integrador")
    SCARED_ROOT  = BASE / "scared_raw"
    EDAM_PATH    = BASE / "Endo-Depth-and-Motion"
    LMSPEC_PATH  = BASE / "EndoLMSPEC"
    IAT_PATH     = BASE / "EndoViT"
    MONOVIT_PATH = BASE / "MonoViT"
    STTN_PATH    = BASE / "Endo-STTN"
    W            = BASE / "scared weights"
    REPO_ROOT    = Path("/content/repo_52")
    if not REPO_ROOT.exists():
        subprocess.check_call(["git","clone","--depth=1",
            "https://github.com/jmtoral/proyecto_integrador_52.git", str(REPO_ROOT)])
    else:
        subprocess.check_call(["git","-C",str(REPO_ROOT),"fetch","origin"])
        subprocess.check_call(["git","-C",str(REPO_ROOT),"reset","--hard","origin/main"])
    SPLIT_FILE = REPO_ROOT / "data" / "splits" / "endovis" / "test_files.txt"
else:
    BASE         = Path("E:/scared_wights_complete/scared weights")
    SCARED_ROOT  = Path("D:/Proyecto_Integrador/Corrreccion_Luz/data/scared_raw")
    EDAM_PATH    = Path("E:/Endo-Depth-and-Motion")
    LMSPEC_PATH  = Path("E:/EndoLMSPEC")
    IAT_PATH     = Path("E:/EndoVit")
    MONOVIT_PATH = Path("E:/MonoViT")
    STTN_PATH    = Path("E:/Endo-STTN")
    W            = BASE
    REPO_ROOT    = Path(r"d:\Proyecto_Integrador\Corrreccion_Luz")
    SPLIT_FILE   = REPO_ROOT / "data" / "splits" / "endovis" / "test_files.txt"

W_MONOIIT      = W / "monoIIT_weights" / "trained-winner-weights"
LMSPEC_WEIGHTS = LMSPEC_PATH / "checkpoint" / "main_net" / "model_256_combined_SSIM5_1.pth"
IAT_WEIGHTS    = IAT_PATH / "Endo4IE" / "best_Epoch50_laplacian_histogan_loss.pth"
STTN_WEIGHTS   = STTN_PATH / "release_model" / "pretrained_model" / "gen_00009.pth"
NPZ_CACHE      = BASE / "split_frames.npz"

def load_split(sf):
    items=[]
    with open(sf) as f:
        for line in f:
            line=line.strip()
            if not line: continue
            folder, fid, _ = line.split()
            ds, kf = folder.split("/")
            items.append(("dataset_"+ds.replace("dataset",""), "keyframe_"+kf.replace("keyframe",""), int(fid)))
    return items
SPLIT_ITEMS = load_split(SPLIT_FILE)
from collections import defaultdict
SPLIT_BY_KF = defaultdict(list)
for ds,kf,fid in SPLIT_ITEMS: SPLIT_BY_KF[(ds,kf)].append(fid)
print(f"{'Colab' if IN_COLAB else 'Local'} | split {len(SPLIT_ITEMS)} frames")

In [ ]:
import numpy as np
def _key(ds,kf,fid): return f"{ds}|{kf}|{fid}"
SPLIT_DATA = {}
_npz = np.load(NPZ_CACHE, allow_pickle=True)
for ds,kf,fid in SPLIT_ITEMS:
    k=_key(ds,kf,fid); ik,gk="img_"+k,"gt_"+k
    if ik in _npz.files:
        gt=_npz[gk] if gk in _npz.files else None
        if gt is not None and gt.size==1 and np.isnan(gt).all(): gt=None
        SPLIT_DATA[k]=(_npz[ik], gt)
def load_split_frame(ds,kf,fid):
    return SPLIT_DATA.get(_key(ds,kf,fid),(None,None))
print(f"Frames en memoria: {len(SPLIT_DATA)}")

In [ ]:
import torch, importlib.util as _ilu, types
import torch.nn as nn
import numpy as np
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

_NETDIR = MONOVIT_PATH / "networks"
for _k in list(sys.modules):
    if _k=="networks" or _k.startswith("networks."): del sys.modules[_k]
_pkg=types.ModuleType("networks"); _pkg.__path__=[str(_NETDIR)]; sys.modules["networks"]=_pkg
def _ls(name,fn):
    sp=_ilu.spec_from_file_location(f"networks.{name}",str(_NETDIR/fn))
    m=_ilu.module_from_spec(sp); sys.modules[f"networks.{name}"]=m; sp.loader.exec_module(m); setattr(_pkg,name,m); return m
_ls("hr_layers","hr_layers.py")
_hr = _ls("hr_decoder","hr_decoder.py")
mpvit_small=_ls("mpvit","mpvit.py").mpvit_small
DepthDecoderHR = _hr.DepthDecoder

enc = mpvit_small(); enc.num_ch_enc=[64,128,216,288,288]
_ed = torch.load(W_MONOIIT/"encoder.pth", map_location=DEVICE)
MH, MW = _ed.get("height",192), _ed.get("width",640)
enc.load_state_dict({k:v for k,v in _ed.items() if k in enc.state_dict()}); enc.to(DEVICE).eval()

_sd = torch.load(W_MONOIIT/"depth.pth", map_location=DEVICE)
dec = DepthDecoderHR()
_r = dec.load_state_dict(_sd, strict=False); dec.to(DEVICE).eval()
print(f"MonoIIT cargado {MH}x{MW} (HR-Depth)")

import cv2, PIL.Image as pil
from torchvision import transforms
def predict_depth(img, max_depth=150.0, min_depth=0.1):
    H,W_=img.shape[:2]
    t=transforms.ToTensor()(pil.fromarray(img).resize((MW,MH),pil.LANCZOS)).unsqueeze(0).to(DEVICE)
    with torch.no_grad(): out=dec(enc(t))
    disp=out[("disp",0)].squeeze().detach().cpu().numpy()
    sd=(1.0/max_depth)+((1.0/min_depth)-(1.0/max_depth))*disp
    sd=cv2.resize(sd,(W_,H)); return 1.0/sd

In [ ]:
import torchvision.transforms as T
subprocess.check_call([sys.executable,"-m","pip","install","-q","IQA_pytorch","path"])

sys.modules["imp"]=types.ModuleType("imp")
_sp=_ilu.spec_from_file_location("IAT_main_a5", IAT_PATH/"experiments"/"model"/"IAT_main.py")
_im=_ilu.module_from_spec(_sp)
if str(IAT_PATH/"experiments") not in sys.path: sys.path.insert(0,str(IAT_PATH/"experiments"))
_sp.loader.exec_module(_im)
iat_net=_im.IAT(in_dim=3, with_global=True, type="exp")
iat_net.load_state_dict(torch.load(IAT_WEIGHTS, map_location=DEVICE)); iat_net.to(DEVICE).eval()

def correct_iat(img):
    t=torch.from_numpy(img.astype(np.float32)/255).permute(2,0,1).unsqueeze(0).to(DEVICE)
    with torch.no_grad(): _,_,e=iat_net(t)
    return (e[0].cpu().clamp(0,1).permute(1,2,0).numpy()*255).astype(np.uint8)

print("IAT listo.")

In [ ]:
FX, FY, CX, CY = 1078.0, 1078.0, 640.0, 512.0

def depth_to_pointcloud(depth_mm, rgb=None, mask=None, stride=2):
    H, W = depth_mm.shape
    uu, vv = np.meshgrid(np.arange(W), np.arange(H))
    if mask is None:
        mask = np.isfinite(depth_mm) & (depth_mm > 0)
    if stride > 1:
        sub = np.zeros_like(mask); sub[::stride, ::stride] = True
        mask = mask & sub
    Z = depth_mm[mask]
    X = (uu[mask] - CX) * Z / FX
    Y = (vv[mask] - CY) * Z / FY
    pts = np.stack([X, Y, Z], axis=1)
    cols = rgb[mask] if rgb is not None else None
    return pts, cols

def median_scale(pred, gt, cap_mm=150.0):
    valid = np.isfinite(gt) & (gt > 0) & (gt < cap_mm)
    if valid.sum() == 0: return pred, valid
    ratio = np.median(gt[valid]) / np.median(pred[valid])
    return pred * ratio, valid

print("Retroproyeccion 3D lista.")

In [ ]:
SCENES_3D = [
    ("under1", "dataset_1", "keyframe_0", 5),
    ("under2", "dataset_2", "keyframe_0", 10),
    ("over1", "dataset_3", "keyframe_0", 8),
    ("over2", "dataset_4", "keyframe_0", 12),
    ("specular1", "dataset_5", "keyframe_0", 7),
    ("specular2", "dataset_6", "keyframe_0", 9),
    ("medium1", "dataset_7", "keyframe_0", 11),
]

print(f"Escenas para 3D: {len(SCENES_3D)}")
for sc_id, ds, kf, fid in SCENES_3D:
    img, gt = load_split_frame(ds, kf, fid)
    if img is not None:
        print(f"  OK {sc_id:10s} -> {ds}/{kf} f{fid}  H={img.shape[0]} W={img.shape[1]}")
    else:
        print(f"  NO {sc_id:10s} -> NO ENCONTRADO")

In [ ]:
import json
import os

STRIDE = 3
OUT_3D_LOCAL = REPO_ROOT / "site" / "demo" / "3d" / "assets"
OUT_3D_GITHUB = REPO_ROOT / "docs" / "demo" / "3d" / "assets"
os.makedirs(OUT_3D_LOCAL, exist_ok=True)
os.makedirs(OUT_3D_GITHUB, exist_ok=True)

def pointcloud_to_json(pts, cols):
    return {
        "vertices": pts.tolist(),
        "colors": (cols.tolist() if cols is not None else [[128,128,128]]*len(pts))
    }

GENERATED = []
for sc_id, ds, kf, fid in SCENES_3D:
    img, gt_mm = load_split_frame(ds, kf, fid)
    if img is None or gt_mm is None:
        print(f"  SKIP {sc_id} (sin imagen o GT)")
        continue

    depth_none = predict_depth(img)
    img_enh = correct_iat(img)
    depth_enh = predict_depth(img_enh)

    depth_none_mm, valid = median_scale(depth_none, gt_mm)
    depth_enh_mm, _ = median_scale(depth_enh, gt_mm)

    pc_none, col_none = depth_to_pointcloud(
        np.where(valid, depth_none_mm, np.nan), rgb=img, mask=valid, stride=STRIDE)
    pc_enh, col_enh = depth_to_pointcloud(
        np.where(valid, depth_enh_mm, np.nan), rgb=img_enh, mask=valid, stride=STRIDE)

    data_none = pointcloud_to_json(pc_none, col_none)
    data_enh = pointcloud_to_json(pc_enh, col_enh)

    fname_none = f"{sc_id}_none.json"
    fname_enh = f"{sc_id}_iat.json"

    for out_dir in [OUT_3D_LOCAL, OUT_3D_GITHUB]:
        with open(out_dir / fname_none, "w") as f:
            json.dump(data_none, f)
        with open(out_dir / fname_enh, "w") as f:
            json.dump(data_enh, f)

    GENERATED.append((sc_id, len(pc_none), len(pc_enh)))
    print(f"  OK {sc_id:10s} | none: {len(pc_none)} pts | iat: {len(pc_enh)} pts")

print(f"Total: {len(GENERATED)} escenas x 2 = {len(GENERATED)*2} JSONs")

## Próximo paso

Las nubes JSON están listas en `site/demo/3d/assets/` y `docs/demo/3d/assets/`.

**App web (HTML + Three.js)**:
1. Dos viewers lado-a-lado (none vs iat).
2. Lupa simultánea en ambos.
3. Rotaciones fijas (frente, arriba, lado).
4. Interactividad: mouse-drag, scroll.

App en `site/demo/3d/index.html`, publicada automáticamente en GitHub Pages.